# LAX — Local Ollama CPU Colab launcher

This notebook runs LAX on a CPU-only runtime using a local Ollama `qwen2.5:1.5b` model for answer generation. It keeps LAX's safety routing and curated BGE/FAISS retrieval. It does not use an Ollama Cloud API key or a per-request hosted API quota.

**Limits and trade-off:** requests are not billed per API call, but CPU/RAM/disk/network/session time still limit usage. This CPU path does not load your trained 1.5B LoRA adapter, because the adapter was trained for the Transformers base model. Your adapter remains unchanged in Google Drive and the GPU launcher can still use it.

## 1. Configure the local Ollama CPU path

No Google Drive mount is required in this mode. Your saved adapter is not modified or deleted.

In [ ]:
from pathlib import Path

# Only needed if you later turn fast mode off; the default fast CPU mode skips it.
ADAPTER_PATH = Path("/content/drive/MyDrive/psychology-qwen-qlora-v2")
print("CPU fast mode enabled: using Qwen2.5-0.5B-Instruct without the LoRA adapter.")
print("The grounded retrieval, topic routing, and high-risk safety check remain enabled.")

## 2. Clone or update the GitHub repository

In [ ]:
from pathlib import Path
import subprocess

REPO_DIR = Path("/content/psychology-raft-rag-chatbot")
REPO_URL = "https://github.com/karthikeyan-crypto/psychology-raft-rag-chatbot.git"

if (REPO_DIR / ".git").exists():
    subprocess.run(
        ["git", "-C", str(REPO_DIR), "pull", "--ff-only", "origin", "main"],
        check=True,
    )
else:
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)

assert (REPO_DIR / "app.py").is_file(), "app.py was not found in the repository."
assert (REPO_DIR / "requirements-colab-cpu.txt").is_file(), "CPU requirements file is missing."
print("Repository is ready:", REPO_DIR)

## 3. Configure CPU and local Ollama environment variables

The Streamlit process inherits these settings and connects to the local Ollama server.

In [ ]:
import os
import sys

os.environ["CUDA_VISIBLE_DEVICES"] = ""
os.environ["LAX_ADAPTER_PATH"] = str(ADAPTER_PATH)
os.environ["LAX_CPU_FAST_MODE"] = "0"
os.environ["LAX_USE_OLLAMA"] = "1"
os.environ["LAX_OLLAMA_BASE_URL"] = "http://127.0.0.1:11434"
os.environ["LAX_OLLAMA_MODEL"] = "qwen2.5:1.5b"
os.environ["LAX_LOAD_GENERAL_ON_CPU"] = "0"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

%cd /content/psychology-raft-rag-chatbot
import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("CPU fast mode: local Ollama")
print("Ollama model:", os.environ["LAX_OLLAMA_MODEL"])
assert not torch.cuda.is_available(), "CUDA is visible. Restart with a CPU runtime and run again."

## 4. Install the CPU-only app dependencies

This requirements file preserves Colab's existing PyTorch installation.

In [ ]:
%pip install -q -r /content/psychology-raft-rag-chatbot/requirements-colab-cpu.txt

## 5. Remove incompatible optional TorchAO

Some Colab runtimes include an older optional TorchAO package that Transformers rejects during model loading. LAX's CPU mode does not use TorchAO, so remove it only when the installed version is `0.16.0` or older.

In [ ]:
import importlib.metadata as metadata
import subprocess
import sys

try:
    torchao_version = metadata.version("torchao")
except metadata.PackageNotFoundError:
    torchao_version = None

print("Existing TorchAO:", torchao_version)
if torchao_version is not None:
    # This notebook is CPU-only. LAX's CPU path does not use TorchAO, and
    # a mismatched optional binary can emit warnings or break model startup.
    print("Removing optional TorchAO from this CPU-only runtime.")
    subprocess.run(
        [sys.executable, "-m", "pip", "uninstall", "-y", "torchao"],
        check=True,
    )
else:
    print("TorchAO is absent; this is expected for CPU-only LAX.")

try:
    print("TorchAO after check:", metadata.version("torchao"))
except metadata.PackageNotFoundError:
    print("TorchAO is not installed (expected for this CPU-only app).")


## 6. Verify installed dependencies

In [ ]:
import importlib.metadata as metadata
import torch

for package in [
    "transformers", "peft", "accelerate",
    "sentence-transformers", "faiss-cpu", "streamlit", "safetensors"
]:
    try:
        print(f"{package}: {metadata.version(package)}")
    except metadata.PackageNotFoundError:
        print(f"MISSING: {package}")

print("PyTorch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())
subprocess.run([sys.executable, "-m", "pip", "check"], check=False)

## 7. Install and start local Ollama

This uses a locally downloaded `qwen2.5:1.5b` model. It does not call Ollama Cloud or a paid hosted LLM API. The first pull downloads around 1 GB; Colab's session time, CPU, RAM, disk and network still apply.

In [ ]:
import json
import os
import shutil
import subprocess
import sys
import time
import urllib.request
from pathlib import Path

OLLAMA_MODEL = os.environ.get("LAX_OLLAMA_MODEL", "qwen2.5:1.5b")
OLLAMA_BASE_URL = os.environ.get("LAX_OLLAMA_BASE_URL", "http://127.0.0.1:11434").rstrip("/")
OLLAMA_LOG = Path("/content/ollama.log")
OLLAMA_PROCESS = None

def run_step(command, label):
    print(f"\n--- {label} ---")
    result = subprocess.run(command, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    print(result.stdout[-12000:] if result.stdout else "(no output)")
    if result.returncode != 0:
        raise RuntimeError(f"{label} failed with exit code {result.returncode}. See the output above.")
    return result

ollama_bin = shutil.which("ollama")
if ollama_bin is None:
    print("Ollama is not installed. Checking required Linux utilities...")
    # The official Linux installer needs curl and, for the current compressed
    # distribution package, zstd. Install these first to avoid a silent pipe failure.
    run_step(["apt-get", "update", "-qq"], "Update apt package list")
    run_step(["apt-get", "install", "-y", "curl", "zstd"], "Install curl and zstd")

    installer_path = Path("/content/ollama-install.sh")
    print("Downloading the official Ollama installer...")
    with urllib.request.urlopen("https://ollama.com/install.sh", timeout=30) as response:
        installer_text = response.read().decode("utf-8")
    if not installer_text.startswith("#!/"):
        raise RuntimeError("Downloaded installer did not look like a shell script. Check network access.")
    installer_path.write_text(installer_text, encoding="utf-8")
    try:
        run_step(["sh", str(installer_path)], "Run official Ollama installer")
    except RuntimeError:
        print("\nOfficial installer failed. Its full captured output is shown above.")
        print("Check that Colab has internet access and enough disk space, then rerun this cell.")
        raise
    ollama_bin = shutil.which("ollama")

if ollama_bin is None:
    for candidate in ["/usr/local/bin/ollama", "/usr/bin/ollama", "/bin/ollama"]:
        if Path(candidate).is_file():
            ollama_bin = candidate
            break
if ollama_bin is None:
    raise RuntimeError("Ollama is still not available after installation. Read the installer output above.")

print("Ollama binary:", ollama_bin)
run_step([ollama_bin, "--version"], "Verify Ollama executable")

def ollama_is_ready():
    try:
        with urllib.request.urlopen(f"{OLLAMA_BASE_URL}/api/tags", timeout=3) as response:
            return response.status == 200
    except Exception:
        return False

if not ollama_is_ready():
    ollama_handle = OLLAMA_LOG.open("w", encoding="utf-8")
    OLLAMA_PROCESS = subprocess.Popen(
        [ollama_bin, "serve"],
        stdout=ollama_handle,
        stderr=subprocess.STDOUT,
        start_new_session=True,
    )
    for _ in range(60):
        if ollama_is_ready():
            break
        if OLLAMA_PROCESS.poll() is not None:
            break
        time.sleep(1)

if not ollama_is_ready():
    print(OLLAMA_LOG.read_text(errors="replace")[-8000:] if OLLAMA_LOG.exists() else "No Ollama log found")
    raise RuntimeError("Ollama did not start on 127.0.0.1:11434. Review /content/ollama.log.")

print("Local Ollama server is ready.")
print(f"Downloading/checking model {OLLAMA_MODEL} (first run can take a few minutes)...")
run_step([ollama_bin, "pull", OLLAMA_MODEL], f"Pull Ollama model {OLLAMA_MODEL}")

with urllib.request.urlopen(f"{OLLAMA_BASE_URL}/api/tags", timeout=5) as response:
    models_payload = json.loads(response.read().decode("utf-8"))
installed_models = [item.get("name", "") for item in models_payload.get("models", [])]
print("Installed local models:", installed_models)
if OLLAMA_MODEL not in installed_models:
    raise RuntimeError(f"Model pull completed but {OLLAMA_MODEL} was not found in /api/tags.")
print("Ollama model verified:", OLLAMA_MODEL)


## 8. Build the local knowledge index if it does not exist

This uses the repository's curated Markdown knowledge files and the BGE-small embedding model.

In [ ]:
import subprocess
index_dir = REPO_DIR / "data" / "vector_index"
index_file = index_dir / "student_wellbeing.faiss"
metadata_file = index_dir / "metadata.json"

if not (index_file.is_file() and metadata_file.is_file()):
    subprocess.run(
        [
            sys.executable, "inference/rag_local.py",
            "--source-dir", "data/sources",
            "--index-dir", "data/vector_index",
        ],
        cwd=str(REPO_DIR),
        check=True,
    )
else:
    print("Existing local FAISS index found; keeping it.")

print("Index ready:", index_file.is_file() and metadata_file.is_file())

## 9. Start Streamlit and inspect its log

The app will call the local Ollama server for psychology answer generation. Keep the Colab runtime active.

In [ ]:
import subprocess
import time
import urllib.request
from pathlib import Path

log_path = Path("/content/lax.log")
log_handle = log_path.open("w")
streamlit_process = subprocess.Popen(
    [
        sys.executable, "-m", "streamlit", "run", "app.py",
        "--server.address=0.0.0.0",
        "--server.port=8501",
        "--server.headless=true",
    ],
    cwd=str(REPO_DIR),
    env=os.environ.copy(),
    stdout=log_handle,
    stderr=subprocess.STDOUT,
    start_new_session=True,
)

print("Streamlit PID:", streamlit_process.pid)
print("Waiting for the local server (the first model load can take time)...")

ready = False
for attempt in range(45):
    time.sleep(2)
    if streamlit_process.poll() is not None:
        print("Streamlit exited with code:", streamlit_process.returncode)
        break
    try:
        with urllib.request.urlopen("http://127.0.0.1:8501/_stcore/health", timeout=3) as response:
            if response.status == 200:
                ready = True
                break
    except Exception:
        pass

print("Local Streamlit health check passed:", ready)
print("\n--- Recent LAX log output ---")
print(log_path.read_text(errors="replace")[-10000:])

if not ready:
    raise RuntimeError("Streamlit did not become ready. Read the log above before retrying.")

## 10. Create a temporary public link using Cloudflare Tunnel

Run this cell last. It keeps the tunnel open while the link is served. Copy the new `https://….trycloudflare.com` URL from the output.

In [ ]:
import re
import shutil
import subprocess
import time
from pathlib import Path
import urllib.request

# Install cloudflared automatically if the earlier install cell was skipped
# or failed. Uses the Debian package and lets apt resolve dependencies.
cloudflared = shutil.which("cloudflared")
if cloudflared is None:
    deb_path = Path("/content/cloudflared-linux-amd64.deb")
    download_url = (
        "https://github.com/cloudflare/cloudflared/releases/latest/download/"
        "cloudflared-linux-amd64.deb"
    )

    print("cloudflared is missing. Downloading the official package...")
    subprocess.run(
        ["wget", "-q", "--show-progress", download_url, "-O", str(deb_path)],
        check=True,
    )

    if not deb_path.is_file() or deb_path.stat().st_size < 1_000_000:
        raise RuntimeError(
            "The cloudflared package download appears incomplete. "
            "Retry this cell or inspect network access."
        )

    print("Installing cloudflared and resolving package dependencies...")
    subprocess.run(
        ["apt-get", "install", "-y", str(deb_path)],
        check=True,
    )
    cloudflared = shutil.which("cloudflared")

if cloudflared is None:
    raise RuntimeError(
        "cloudflared still was not found after installation. "
        "Run !dpkg -l cloudflared and share the output."
    )

version = subprocess.run(
    [cloudflared, "--version"],
    capture_output=True,
    text=True,
    check=True,
)
print("Installed:", version.stdout.strip())

# Confirm Streamlit is answering locally before opening the public tunnel.
try:
    with urllib.request.urlopen("http://127.0.0.1:8501/_stcore/health", timeout=5) as response:
        if response.status != 200:
            raise RuntimeError(f"Streamlit health check returned HTTP {response.status}")
except Exception as exc:
    raise RuntimeError(
        "Streamlit is not healthy at http://127.0.0.1:8501. "
        "Run the Streamlit startup cell and inspect /content/lax.log before starting the tunnel."
    ) from exc

log_path = Path("/content/cloudflared.log")
log_path.write_text("", encoding="utf-8")
log_handle = log_path.open("a", encoding="utf-8")

cloudflared_process = subprocess.Popen(
    [
        cloudflared, "tunnel",
        "--url", "http://127.0.0.1:8501",
        "--protocol", "http2",
        "--loglevel", "info",
    ],
    stdout=log_handle,
    stderr=subprocess.STDOUT,
    text=True,
    start_new_session=True,
)

print("Cloudflare Tunnel started in the background.")
print("Process ID:", cloudflared_process.pid)
print("Checking for a public URL (up to 120 seconds)...")

public_url = None
deadline = time.time() + 120
while time.time() < deadline:
    if cloudflared_process.poll() is not None:
        break

    logs = log_path.read_text(encoding="utf-8", errors="replace")
    found = re.findall(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", logs)
    if found:
        public_url = found[-1]
        break
    time.sleep(2)

if public_url:
    print()
    print("LAX public URL:", public_url)
    print("Open the URL in a new browser tab.")
    print("Keep this Colab runtime active while using LAX.")
else:
    logs = log_path.read_text(encoding="utf-8", errors="replace")
    print("No public URL appeared within 120 seconds.")
    print()
    print("--- Last Cloudflare logs ---")
    print(logs[-12000:] if logs else "(Cloudflare log is empty)")
    if cloudflared_process.poll() is not None:
        print("Cloudflare exited with code:", cloudflared_process.returncode)
    print()
    print("If logs show connection refused to 127.0.0.1:8501, restart Streamlit first.")
    print("For DNS or network errors, retry on a working network.")

## Troubleshooting

- **Ollama server error:** inspect `/content/ollama.log`; the service must answer at `http://127.0.0.1:11434/api/tags`.
- **Model not installed:** run `!ollama pull qwen2.5:1.5b` and verify it appears in `!ollama list`.
- **Host error:** inspect `/content/lax.log` and confirm the local Streamlit health check passes before opening the tunnel.
- **Slow first response:** the first run downloads the Ollama model and BGE embedding model. Later replies still depend on available CPU and RAM.
- **Saved adapter:** this CPU Ollama path does not load or modify your trained LoRA adapter. The GPU launcher can still use the saved adapter when a compatible GPU runtime is available.

LAX is a student-wellbeing psychoeducation assistant, not a therapist or emergency service.